In [4]:
import awswrangler as wr
import pandas as pd

# Define S3 path
s3_path = "s3://thesis--ec331-s3/FCAS_RAISE1SEC-Volume-Bids/PUBLIC_DVD_BIDPEROFFER1_202311010000.parquet/part.131.parquet"

# Read the Parquet file
df = wr.s3.read_parquet(path=s3_path)

# Display the first few rows
print(df.head())

   I  BIDS  BIDOFFERPERIOD    1      DUID    BIDTYPE          TRADINGDATE  \
0  D  BIDS  BIDOFFERPERIOD  1.0  ASRMGE01  RAISE1SEC  2023/11/12 00:00:00   
1  D  BIDS  BIDOFFERPERIOD  1.0  ASRMGE01  RAISE1SEC  2023/11/12 00:00:00   
2  D  BIDS  BIDOFFERPERIOD  1.0  ASRMGE01  RAISE1SEC  2023/11/12 00:00:00   
3  D  BIDS  BIDOFFERPERIOD  1.0  ASRMGE01  RAISE1SEC  2023/11/12 00:00:00   
4  D  BIDS  BIDOFFERPERIOD  1.0  ASRMGE01  RAISE1SEC  2023/11/12 00:00:00   

         OFFERDATETIME  PERIODID  MAXAVAIL  ...  BANDAVAIL2  BANDAVAIL3  \
0  2023/11/11 16:38:23     113.0       1.0  ...         0.0         0.0   
1  2023/11/11 16:38:23     114.0       1.0  ...         0.0         0.0   
2  2023/11/11 16:38:23     115.0       1.0  ...         0.0         0.0   
3  2023/11/11 16:38:23     116.0       1.0  ...         0.0         0.0   
4  2023/11/11 16:38:23     117.0       1.0  ...         0.0         0.0   

   BANDAVAIL4  BANDAVAIL5  BANDAVAIL6  BANDAVAIL7  BANDAVAIL8  BANDAVAIL9  \
0        

In [1]:
# Check for dupes
#!/usr/bin/env python3
"""
Script to check for duplicates in RAISE1SEC parquet files on S3.
"""

import awswrangler as wr
import pandas as pd
import gc
import logging
import time
import boto3
import os
import psutil
from datetime import datetime

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("duplicate_check.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# File path
SOURCE_FILE_PATH = "s3://thesis--ec331-s3/volume-bids-compressed/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER1_202310010000.parquet/combined.parquet"

# Columns to identify duplicates
DUPLICATE_KEYS = ["TRADINGDATE", "DUID", "PERIODID"]

# Column for determining latest offer (if present)
OFFER_DATE_COLUMN = "OFFERDATETIME"
FALLBACK_OFFER_COLUMNS = ["LASTCHANGED", "SETTLEMENTDATE"]

def log_memory_usage(context=""):
    """Log current memory usage."""
    memory = psutil.virtual_memory()
    logger.info(f"Memory usage {context}: {memory.percent:.1f}% (Used: {memory.used / (1024**3):.2f} GB)")

def check_duplicates(file_path, duplicate_keys):
    """Check for duplicates in a single parquet file and return statistics."""
    start_time = time.time()
    
    logger.info(f"Analyzing file: {file_path}")
    log_memory_usage("before processing")
    
    try:
        # Check if file exists
        file_exists = wr.s3.does_object_exist(path=file_path)
        if not file_exists:
            logger.warning(f"File does not exist: {file_path}")
            return {
                'success': False,
                'error': 'File does not exist'
            }
        
        # Read the file
        df = wr.s3.read_parquet(path=file_path)
        
        if df.empty:
            logger.info(f"File is empty: {file_path}")
            return {
                'success': True,
                'original_rows': 0,
                'duplicate_count': 0,
                'duplicate_percent': 0,
                'available_keys': []
            }
        
        original_count = len(df)
        logger.info(f"Read {original_count:,} rows from {file_path}")
        
        # Check column presence and data types
        logger.info(f"Columns in the dataset: {df.columns.tolist()}")
        
        # Convert date columns if present
        if "TRADINGDATE" in df.columns:
            df["TRADINGDATE"] = pd.to_datetime(df["TRADINGDATE"], errors="coerce")
            logger.info(f"Converted TRADINGDATE to datetime")
        
        # Check which duplicate keys are available
        available_keys = [key for key in duplicate_keys if key in df.columns]
        logger.info(f"Available duplicate keys: {available_keys}")
        
        if not available_keys:
            logger.warning(f"No duplicate keys available in {file_path}")
            return {
                'success': True,
                'original_rows': original_count,
                'duplicate_count': 0,
                'duplicate_percent': 0,
                'available_keys': []
            }
        
        # Handle NaN values in the duplicate keys
        for key in available_keys:
            if key == "TRADINGDATE":
                df[key] = df[key].fillna(pd.Timestamp('1970-01-01'))
            elif pd.api.types.is_numeric_dtype(df[key]):
                df[key] = df[key].fillna(-999)
            else:
                df[key] = df[key].fillna("UNKNOWN")
        
        # Sample data
        logger.info("Sample data (first 5 rows):")
        logger.info(df.head(5).to_string())
        
        # Check for duplicates
        duplicates = df.duplicated(subset=available_keys, keep=False)
        duplicate_count = duplicates.sum()
        duplicate_percent = (duplicate_count / original_count * 100)
        
        logger.info(f"Found {duplicate_count:,} duplicate rows ({duplicate_percent:.2f}%)")
        
        # If duplicates exist, analyze them further
        if duplicate_count > 0:
            # Get duplicate rows
            duplicate_rows = df[duplicates].copy()
            
            # Check for offer date columns
            offer_date_col = None
            if OFFER_DATE_COLUMN in df.columns:
                offer_date_col = OFFER_DATE_COLUMN
            else:
                for col in FALLBACK_OFFER_COLUMNS:
                    if col in df.columns:
                        offer_date_col = col
                        break
            
            if offer_date_col:
                logger.info(f"Found offer date column: {offer_date_col}")
                
                # Convert to datetime if needed
                if offer_date_col in df.columns:
                    duplicate_rows[offer_date_col] = pd.to_datetime(duplicate_rows[offer_date_col], errors="coerce")
                
                # Sample duplicates
                logger.info("Sample of duplicate data:")
                for key_group, group_df in duplicate_rows.groupby(available_keys):
                    if isinstance(key_group, tuple):
                        key_str = " | ".join([str(k) for k in key_group])
                    else:
                        key_str = str(key_group)
                    
                    logger.info(f"Duplicate group with key: {key_str}")
                    logger.info(f"Group has {len(group_df)} duplicates")
                    
                    # Sort by offer date if available
                    if offer_date_col in group_df.columns:
                        group_df = group_df.sort_values(by=offer_date_col, ascending=False)
                    
                    logger.info(group_df.head(2).to_string())
                    
                    # Only show first duplicate group as an example
                    break
        
        # Summary
        execution_time = time.time() - start_time
        logger.info(f"Analysis completed in {execution_time:.2f} seconds")
        
        return {
            'success': True,
            'original_rows': original_count,
            'duplicate_count': duplicate_count,
            'duplicate_percent': duplicate_percent,
            'available_keys': available_keys,
            'execution_time': execution_time
        }
    
    except Exception as e:
        logger.error(f"Error analyzing {file_path}: {str(e)}", exc_info=True)
        return {
            'success': False,
            'error': str(e)
        }
    finally:
        # Clean up
        if 'df' in locals():
            del df
            gc.collect()
        log_memory_usage("after processing")

def main():
    """Main function to check duplicates in the specified file."""
    start_time = time.time()
    
    try:
        logger.info(f"Starting duplicate analysis")
        logger.info(f"File: {SOURCE_FILE_PATH}")
        logger.info(f"Duplicate keys: {DUPLICATE_KEYS}")
        
        # Check duplicate keys
        result = check_duplicates(SOURCE_FILE_PATH, DUPLICATE_KEYS)
        
        if result['success']:
            logger.info("\n" + "="*80)
            logger.info(f"DUPLICATE ANALYSIS SUMMARY")
            logger.info("="*80)
            logger.info(f"File: {SOURCE_FILE_PATH}")
            logger.info(f"Total rows: {result.get('original_rows', 0):,}")
            logger.info(f"Duplicate rows: {result.get('duplicate_count', 0):,}")
            logger.info(f"Duplicate percentage: {result.get('duplicate_percent', 0):.2f}%")
            logger.info(f"Keys used: {result.get('available_keys', [])}")
            logger.info(f"Execution time: {result.get('execution_time', 0):.2f} seconds")
            logger.info("="*80)
        else:
            logger.error(f"Analysis failed: {result.get('error', 'Unknown error')}")
        
    except Exception as e:
        logger.error(f"Error in main process: {str(e)}", exc_info=True)
    finally:
        total_time = time.time() - start_time
        logger.info(f"Total execution time: {total_time:.2f} seconds")
        gc.collect()

if __name__ == "__main__":
    main()


2025-03-19 20:27:12,104 - INFO - Starting duplicate analysis
2025-03-19 20:27:12,105 - INFO - File: s3://thesis--ec331-s3/volume-bids-compressed/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER1_202310010000.parquet/combined.parquet
2025-03-19 20:27:12,106 - INFO - Duplicate keys: ['TRADINGDATE', 'DUID', 'PERIODID']
2025-03-19 20:27:12,107 - INFO - Analyzing file: s3://thesis--ec331-s3/volume-bids-compressed/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER1_202310010000.parquet/combined.parquet
2025-03-19 20:27:12,109 - INFO - Memory usage before processing: 51.1% (Used: 31.25 GB)
2025-03-19 20:27:12,130 - INFO - Found credentials from IAM Role: thesis
2025-03-19 20:27:12,440 - INFO - Found credentials from IAM Role: thesis
2025-03-19 20:27:38,284 - INFO - Read 16,966,080 rows from s3://thesis--ec331-s3/volume-bids-compressed/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER1_202310010000.parquet/combined.parquet
2025-03-19 20:27:38,285 - INFO - Columns in the dataset: ['I', 'BIDS', 'BIDOFFERPERIOD', '1', 'DUID', 'BIDTYPE', 'TRADINGD

In [1]:
#!/usr/bin/env python3
"""
Script to deduplicate a single RAISE1SEC parquet file on S3 by keeping only the latest offer date
for each combination of TRADINGDATE, DUID, and PERIODID, while preserving the original file structure.
"""

import awswrangler as wr
import pandas as pd
import gc
import logging
import time
import boto3
import os
import psutil
from datetime import datetime

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("deduplicate_preserve_structure.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# S3 file path to process
SOURCE_PATH = "s3://thesis--ec331-s3/volume-bids-compressed/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER1_202310010000.parquet/combined.parquet"
# Set target path. In this example, we preserve the folder structure and write to "deduped.parquet"
TARGET_PATH = "s3://thesis--ec331-s3/de-duped-volume-bids/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER1_202310010000.parquet/deduped.parquet"

# Columns to identify duplicates
DUPLICATE_KEYS = ["TRADINGDATE", "DUID", "PERIODID"]

# Column for determining latest offer
OFFER_DATE_COLUMN = "OFFERDATETIME"
FALLBACK_OFFER_COLUMNS = ["LASTCHANGED", "SETTLEMENTDATE"]

# Processing parameters
DRY_RUN = False  # Set to True to analyze without writing

def log_memory_usage(context=""):
    """Log current memory usage."""
    memory = psutil.virtual_memory()
    logger.info(f"Memory usage {context}: {memory.percent:.1f}% (Used: {memory.used / (1024**3):.2f} GB)")

def parse_s3_path(s3_path):
    """Parse an S3 path into bucket and prefix."""
    if not s3_path.startswith('s3://'):
        raise ValueError(f"Invalid S3 path: {s3_path}")
    
    path = s3_path[5:]
    parts = path.split('/', 1)
    bucket = parts[0]
    prefix = parts[1] if len(parts) > 1 else ''
    
    return bucket, prefix

def check_s3_write_access(s3_client, bucket_name):
    """Test if we can write to the target S3 bucket."""
    try:
        test_key = f"_test_write_access_{datetime.now().strftime('%Y%m%d%H%M%S')}.txt"
        s3_client.put_object(
            Bucket=bucket_name,
            Key=test_key,
            Body="Test write access"
        )
        s3_client.delete_object(
            Bucket=bucket_name,
            Key=test_key
        )
        return True
    except Exception as e:
        logger.error(f"Failed to write to S3 bucket {bucket_name}: {str(e)}")
        return False

def deduplicate_file(source_path, target_path, duplicate_keys, offer_date_column, dry_run=False):
    """Deduplicate a parquet file keeping only the latest offer date for each key combination."""
    start_time = time.time()
    
    logger.info(f"Processing: {source_path} -> {target_path}")
    log_memory_usage("before processing")
    
    try:
        # Check if file exists
        file_exists = wr.s3.does_object_exist(path=source_path)
        if not file_exists:
            logger.warning(f"File does not exist: {source_path}")
            return {
                'success': False,
                'error': 'File does not exist',
                'original_rows': 0,
                'deduplicated_rows': 0,
                'duplicates_removed': 0
            }
        
        # Read the file
        logger.info(f"Reading parquet file from S3")
        df = wr.s3.read_parquet(path=source_path)
        
        if df.empty:
            logger.info(f"File is empty: {source_path}")
            if not dry_run:
                wr.s3.to_parquet(
                    df=df,
                    path=target_path,
                    index=False,
                    compression='snappy'
                )
            return {
                'success': True,
                'original_rows': 0,
                'deduplicated_rows': 0,
                'duplicates_removed': 0
            }
        
        original_count = len(df)
        logger.info(f"Read {original_count:,} rows from {source_path}")
        
        # Convert date columns
        if "TRADINGDATE" in df.columns:
            df["TRADINGDATE"] = pd.to_datetime(df["TRADINGDATE"], errors="coerce")
            logger.info("Converted TRADINGDATE to datetime")
        
        # Check which duplicate keys are available
        available_keys = [key for key in duplicate_keys if key in df.columns]
        logger.info(f"Available duplicate keys: {available_keys}")
        
        if not available_keys:
            logger.warning(f"No duplicate keys available in {source_path}")
            if not dry_run:
                wr.s3.to_parquet(
                    df=df,
                    path=target_path,
                    index=False,
                    compression='snappy'
                )
            return {
                'success': True,
                'original_rows': original_count,
                'deduplicated_rows': original_count,
                'duplicates_removed': 0
            }
        
        # Handle NaN values in the duplicate keys
        for key in available_keys:
            if key == "TRADINGDATE":
                df[key] = df[key].fillna(pd.Timestamp('1970-01-01'))
            elif pd.api.types.is_numeric_dtype(df[key]):
                df[key] = df[key].fillna(-999)
            else:
                df[key] = df[key].fillna("UNKNOWN")
        
        # Identify offer date column for sorting
        offer_date_col = None
        if offer_date_column in df.columns:
            offer_date_col = offer_date_column
            logger.info(f"Using offer date column: {offer_date_col}")
        else:
            # Try fallback columns
            for col in FALLBACK_OFFER_COLUMNS:
                if col in df.columns:
                    offer_date_col = col
                    logger.info(f"Using fallback offer date column: {col}")
                    break
        
        if not offer_date_col:
            logger.warning("No offer date column found, can't determine latest offer")
            return {
                'success': False,
                'error': 'No offer date column found',
                'original_rows': original_count,
                'deduplicated_rows': 0,
                'duplicates_removed': 0
            }
        
        # Convert offer date to datetime
        df[offer_date_col] = pd.to_datetime(df[offer_date_col], errors="coerce")
        
        # Check for duplicates
        duplicates_exist = df.duplicated(subset=available_keys, keep=False).any()
        
        if duplicates_exist:
            logger.info(f"Found duplicates, sorting by {offer_date_col} to keep latest")
            # Sort by offer date (descending) and drop duplicates, keeping the first occurrence (latest date)
            df_sorted = df.sort_values(by=[*available_keys, offer_date_col],
                                       ascending=[True, True, True, False])
            df_deduped = df_sorted.drop_duplicates(subset=available_keys, keep='first')
            
            deduplicated_count = len(df_deduped)
            removed_count = original_count - deduplicated_count
            duplicates_percent = (removed_count / original_count * 100)
            
            logger.info(f"Removed {removed_count:,} duplicates ({duplicates_percent:.2f}%)")
            logger.info(f"Final dataset has {deduplicated_count:,} rows")
            
            # Write deduplicated data
            if not dry_run:
                logger.info(f"Writing {deduplicated_count:,} rows to {target_path}")
                wr.s3.to_parquet(
                    df=df_deduped,
                    path=target_path,
                    index=False,
                    compression='snappy'
                )
                logger.info(f"Successfully wrote deduplicated data to {target_path}")
            else:
                logger.info(f"DRY RUN: Would write {deduplicated_count:,} rows to {target_path}")
            
            return {
                'success': True,
                'original_rows': original_count,
                'deduplicated_rows': deduplicated_count,
                'duplicates_removed': removed_count,
                'duplicates_percent': duplicates_percent,
                'offer_date_used': offer_date_col,
                'execution_time': time.time() - start_time
            }
        else:
            logger.info("No duplicates found in the file")
            if not dry_run:
                wr.s3.to_parquet(
                    df=df,
                    path=target_path,
                    index=False,
                    compression='snappy'
                )
                logger.info(f"Successfully wrote data to {target_path}")
            else:
                logger.info(f"DRY RUN: Would write {original_count:,} rows to {target_path}")
            
            return {
                'success': True,
                'original_rows': original_count,
                'deduplicated_rows': original_count,
                'duplicates_removed': 0,
                'duplicates_percent': 0,
                'offer_date_used': None,
                'execution_time': time.time() - start_time
            }
    
    except Exception as e:
        logger.error(f"Error processing {source_path}: {str(e)}", exc_info=True)
        return {
            'success': False,
            'error': str(e),
            'original_rows': 0,
            'deduplicated_rows': 0,
            'duplicates_removed': 0
        }
    finally:
        # Clean up
        if 'df' in locals():
            del df
        if 'df_sorted' in locals():
            del df_sorted
        gc.collect()

def main():
    """Main function to deduplicate a single file."""
    overall_start_time = time.time()
    
    logger.info("Starting deduplication process on a single file")
    logger.info(f"Source file: {SOURCE_PATH}")
    logger.info(f"Target file: {TARGET_PATH}")
    logger.info(f"Duplicate keys: {DUPLICATE_KEYS}")
    logger.info(f"Offer date column: {OFFER_DATE_COLUMN}")
    logger.info(f"Dry run: {DRY_RUN}")
    
    log_memory_usage("start")
    
    # Create S3 client and check write access for target bucket
    s3_client = boto3.client('s3')
    target_bucket, _ = parse_s3_path(TARGET_PATH)
    if not DRY_RUN and not check_s3_write_access(s3_client, target_bucket):
        logger.error(f"Cannot write to target bucket: {target_bucket}")
        return
    
    # Process the single file
    result = deduplicate_file(SOURCE_PATH, TARGET_PATH, DUPLICATE_KEYS, OFFER_DATE_COLUMN, DRY_RUN)
    logger.info(f"Deduplication result: {result}")
    
    overall_execution_time = time.time() - overall_start_time
    logger.info(f"Total execution time: {overall_execution_time:.2f} seconds")
    log_memory_usage("end")

if __name__ == "__main__":
    main()

2025-03-20 08:28:15,699 - INFO - Starting deduplication process on a single file
2025-03-20 08:28:15,700 - INFO - Source file: s3://thesis--ec331-s3/volume-bids-compressed/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER1_202310010000.parquet/combined.parquet
2025-03-20 08:28:15,701 - INFO - Target file: s3://thesis--ec331-s3/de-duped-volume-bids/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER1_202310010000.parquet/deduped.parquet
2025-03-20 08:28:15,701 - INFO - Duplicate keys: ['TRADINGDATE', 'DUID', 'PERIODID']
2025-03-20 08:28:15,702 - INFO - Offer date column: OFFERDATETIME
2025-03-20 08:28:15,703 - INFO - Dry run: False
2025-03-20 08:28:15,704 - INFO - Memory usage start: 18.0% (Used: 10.62 GB)
2025-03-20 08:28:15,725 - INFO - Found credentials from IAM Role: thesis
2025-03-20 08:28:16,035 - INFO - Processing: s3://thesis--ec331-s3/volume-bids-compressed/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER1_202310010000.parquet/combined.parquet -> s3://thesis--ec331-s3/de-duped-volume-bids/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER1_202310